
# 002_decode_msaa_condition_specific_appendable.ipynb

Flexible appendable decoding notebook for MS-AA outputs.

This notebook:

- loads saved MS-AA `.npz` fit files
- runs decoding for one requested configuration
- appends results to summary CSVs
- skips rows that already exist
- optionally overwrites existing rows if requested

It supports:

- `ANALYSIS_TYPE = "spatial"` or `"temporal"`
- `FIT_SCOPE = "within"` or `"across"`

Outputs:

- `full_summary.csv`
- `topm_summary.csv`
- `per_archetype_mean_accuracy.csv`
- `rankings.npy`

The goal is that you can run this notebook repeatedly as you finish fits, without losing previous decoding results.


**Branch change:** across-condition top-m rankings are now condition-specific: `rankings[(condition, K)]`, not averaged across conditions.

Outputs are written to `msaa_condrank_decoding_outputs_{analysis_type}_{fit_scope}` so they do not overwrite the older global-ranking outputs.


In [ ]:
from pathlib import Path
import re
from collections import defaultdict

BASE_DIR = Path("msaa_flexible_outputs_npz")  # <-- change this

k_dict = defaultdict(set)

for f in BASE_DIR.rglob("*.npz"):
    
    # extract K
    match = re.search(r"K[_=]?(\d+)", f.name)
    if not match:
        continue
    
    K = int(match.group(1))
    
    # infer metadata from filename
    name = f.name.lower()
    
    if "spatial" in name:
        analysis = "spatial"
    elif "temporal" in name:
        analysis = "temporal"
    else:
        analysis = "unknown"
    
    if "within" in name:
        scope = "within"
    elif "across" in name:
        scope = "across"
    else:
        scope = "unknown"
    
    key = (analysis, scope)
    k_dict[key].add(K)

# print nicely
for key, ks in k_dict.items():
    print(f"{key}: {sorted(ks)}")

In [ ]:
from pathlib import Path

# ============================================================
# User settings
# ============================================================

ANALYSIS_TYPE = "temporal"   # "spatial" or "temporal"
FIT_SCOPE = "across"        # "within" or "across"

FIT_LOAD_DIR = "msaa_flexible_outputs_npz"
DECODE_OUTPUT_DIR = f"msaa_condrank_decoding_outputs_{ANALYSIS_TYPE}_{FIT_SCOPE}"

CONDITIONS = ["intact", "word", "rest"]
K_VALUES = [
    2, 3, 5, 6, 9, 11, 15, 17, 18, 20,
    23, 25, 27, 30, 38, 45,
 54, 60, 75, 81, 90, 108, 100, 200,
    300,
]


# Top-m decoding values
TOP_M_VALUES = [1, 2, 3, 5, 10]

# Decoding parameters
NFOLDS_DECODE = 2
NREPS_DECODE = 20
RNG_SEED = 42

# Append behavior
APPEND_MODE = True
OVERWRITE_EXISTING_ROWS = False

# Safer when debugging: rebuild rankings from the current per-archetype CSV rather than keeping stale entries.
FORCE_REBUILD_RANKINGS = True

# Add source fit filename/hash-like diagnostics to result rows.
RECORD_FIT_SOURCE = True

# What to decode
# Set RUN_DECODING=False to skip computation and only make figures/audits from existing outputs.
RUN_DECODING = False
RUN_FIGURES = True
RUN_FULL_RECONSTRUCTION = True
RUN_PER_ARCHETYPE = True
RUN_TOP_M = True

# For speed, you can limit per-archetype decoding to a subset.
# None = all archetypes.
ARCHETYPE_INDICES_TO_DECODE = None
# Example: ARCHETYPE_INDICES_TO_DECODE = list(range(50))

# File naming expected from fit notebook
# Across:
#   spatialAA_across_acrossCond_K{K}.npz
#   temporalAA_across_acrossCond_K{K}.npz
# Within:
#   spatialAA_within_{condition}_K{K}.npz
#   temporalAA_within_{condition}_K{K}.npz

print("Configuration:")
print("  ANALYSIS_TYPE:", ANALYSIS_TYPE)
print("  FIT_SCOPE:", FIT_SCOPE)
print("  FIT_LOAD_DIR:", FIT_LOAD_DIR)
print("  DECODE_OUTPUT_DIR:", DECODE_OUTPUT_DIR)

In [ ]:

# ============================================================
# Imports
# ============================================================

%matplotlib inline

import os
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
import matplotlib.pyplot as plt

Path(DECODE_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

FULL_SUMMARY_CSV = os.path.join(DECODE_OUTPUT_DIR, "full_summary.csv")
TOPM_SUMMARY_CSV = os.path.join(DECODE_OUTPUT_DIR, "topm_summary.csv")
PER_ARCHETYPE_CSV = os.path.join(DECODE_OUTPUT_DIR, "per_archetype_mean_accuracy.csv")
RANKINGS_NPY = os.path.join(DECODE_OUTPUT_DIR, "rankings.npy")

print("Ready.")

In [ ]:
# ============================================================
# FIGURE SAVING SETTINGS -- EXPLICIT, NO RECURSION
# ============================================================

from pathlib import Path

SAVE_FIGS = True
FIG_ROOT = "/Users/lowen/Desktop/papers/archetypes/figures"
FIG_NOTEBOOK_DIR = "002_decode_msaa_appendable"
FIG_DIR = Path(FIG_ROOT) / FIG_NOTEBOOK_DIR
FIG_FORMAT = "pdf"   # "pdf", "png", or "svg"
DPI = 300

FIG_DIR.mkdir(parents=True, exist_ok=True)

_fig_counter = 0

def _sanitize_fig_name(name):
    name = str(name).replace(" ", "_").replace("|", "_").replace("/", "-").replace("\\", "-")
    name = "".join(ch for ch in name if ch.isalnum() or ch in ["_", "-", "."])
    return name[:160] if name else "figure"

def _figure_has_content(fig=None):
    if fig is None:
        fig = plt.gcf()
    if len(fig.axes) == 0:
        return False
    for ax in fig.axes:
        if ax.lines or ax.collections or ax.images or ax.patches or ax.texts or ax.get_title():
            return True
    return True

def save_current_fig(name=None):
    global _fig_counter
    if not SAVE_FIGS:
        return None
    fig = plt.gcf()
    if not _figure_has_content(fig):
        return None
    _fig_counter += 1
    if name is None:
        try:
            title = plt.gca().get_title()
        except Exception:
            title = ""
        label = _sanitize_fig_name(title if title else "figure")
    else:
        label = _sanitize_fig_name(name)
    out = FIG_DIR / f"{_fig_counter:03d}_{label}.{FIG_FORMAT}"
    fig.savefig(out, dpi=DPI, bbox_inches="tight")
    print("Saved:", out)
    return out

def savefig(name=None, force=True):
    return save_current_fig(name=name)

print("Figure directory:", FIG_DIR)
print("Explicit save mode: plt.show is not patched.")


In [ ]:
# ============================================================
# CACHE SETTINGS
# ============================================================

from pathlib import Path
import pickle

CACHE_DIR = Path("002_decode_msaa_appendable_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

USE_CACHE = True
OVERWRITE_CACHE = False

CLUSTER_SUMMARY_CACHE = CACHE_DIR / "cluster_summary_df.csv"
SELECTED_ARCHETYPES_CACHE = CACHE_DIR / "selected_archetypes_dict.npy"
PLOT_DATA_CACHE = CACHE_DIR / "plot_data_cache.pkl"

print("Cache directory:", CACHE_DIR.resolve())
print("USE_CACHE:", USE_CACHE)
print("OVERWRITE_CACHE:", OVERWRITE_CACHE)

## Append / update helpers

In [ ]:

FULL_KEY_COLS = ["analysis_type", "fit_scope", "condition", "K"]
TOPM_KEY_COLS = ["analysis_type", "fit_scope", "condition", "K", "top_m"]
PER_ARCHETYPE_KEY_COLS = ["analysis_type", "fit_scope", "condition", "K", "archetype"]


def load_existing_csv(path):
    if os.path.exists(path):
        return pd.read_csv(path)
    return pd.DataFrame()


def ensure_metadata_columns(df, analysis_type=None, fit_scope=None):
    """
    Backward compatibility helper.

    Older decoding CSVs may not have analysis_type / fit_scope columns.
    Since each output directory is already specific to one analysis_type and fit_scope,
    we can safely add those columns when missing.
    """
    df = df.copy()

    if analysis_type is None:
        analysis_type = ANALYSIS_TYPE
    if fit_scope is None:
        fit_scope = FIT_SCOPE

    if "analysis_type" not in df.columns:
        df["analysis_type"] = analysis_type

    if "fit_scope" not in df.columns:
        df["fit_scope"] = fit_scope

    return df


def ensure_key_columns(df, key_cols, analysis_type=None, fit_scope=None):
    """
    Make sure all key columns exist before indexing.

    This prevents errors like:
      KeyError: "['analysis_type', 'fit_scope'] not in index"

    If a non-metadata key is missing, that usually means the file is not compatible
    with this result type, so we raise a clear error.
    """
    df = ensure_metadata_columns(df, analysis_type=analysis_type, fit_scope=fit_scope)

    missing = [c for c in key_cols if c not in df.columns]
    if missing:
        raise KeyError(
            f"Existing CSV is missing required key columns {missing}. "
            f"Available columns are {list(df.columns)}. "
            f"Path may correspond to a different result type, or the old CSV may need migration."
        )

    return df


def normalize_key_columns(df, key_cols):
    df = df.copy()
    for col in key_cols:
        if col in df.columns:
            if col in ["K", "top_m", "archetype"]:
                df[col] = df[col].astype(int)
            else:
                df[col] = df[col].astype(str)
    return df


def append_or_update_csv(new_df, path, key_cols, overwrite_existing=False):
    new_df = new_df.copy()

    if len(new_df) == 0:
        print(f"No new rows to write for {path}")
        return load_existing_csv(path)

    existing = load_existing_csv(path)

    # Ensure current new rows always have metadata columns
    new_df = ensure_metadata_columns(new_df)

    # Existing rows may come from an older notebook and lack metadata columns
    if len(existing):
        existing = ensure_key_columns(existing, key_cols)

    new_df = ensure_key_columns(new_df, key_cols)

    new_df = normalize_key_columns(new_df, key_cols)
    existing = normalize_key_columns(existing, key_cols) if len(existing) else existing

    if len(existing) == 0:
        combined = new_df.copy()
        skipped = 0
        appended = len(new_df)
    else:
        existing_keys = set(map(tuple, existing[key_cols].astype(str).values))
        new_keys = set(map(tuple, new_df[key_cols].astype(str).values))

        if overwrite_existing:
            keep_mask = ~existing[key_cols].astype(str).apply(tuple, axis=1).isin(new_keys)
            combined = pd.concat([existing.loc[keep_mask], new_df], ignore_index=True)
            skipped = 0
            appended = len(new_df)
        else:
            append_mask = ~new_df[key_cols].astype(str).apply(tuple, axis=1).isin(existing_keys)
            rows_to_append = new_df.loc[append_mask].copy()
            combined = pd.concat([existing, rows_to_append], ignore_index=True)
            skipped = len(new_df) - len(rows_to_append)
            appended = len(rows_to_append)

    sort_cols = [c for c in key_cols if c in combined.columns]
    if sort_cols:
        combined = combined.sort_values(sort_cols).reset_index(drop=True)

    combined.to_csv(path, index=False)
    print(f"{os.path.basename(path)}: appended/replaced {appended}, skipped {skipped}, total {len(combined)}")

    return combined


def existing_keys(path, key_cols):
    df = load_existing_csv(path)
    if len(df) == 0:
        return set()

    df = ensure_key_columns(df, key_cols)
    df = normalize_key_columns(df, key_cols)

    return set(map(tuple, df[key_cols].astype(str).values))


def should_run_config(path, key_cols, key_values, overwrite_existing=False):
    if overwrite_existing:
        return True

    keys = existing_keys(path, key_cols)

    # Make sure key_values has metadata too
    key_values = dict(key_values)
    key_values.setdefault("analysis_type", ANALYSIS_TYPE)
    key_values.setdefault("fit_scope", FIT_SCOPE)

    key_tuple = tuple(str(key_values[col]) for col in key_cols)
    return key_tuple not in keys


def load_rankings(path):
    if os.path.exists(path):
        return np.load(path, allow_pickle=True).item()
    return {}


def save_or_update_rankings(new_rankings, path, overwrite_existing=False):
    rankings = load_rankings(path)

    for key, value in new_rankings.items():
        if overwrite_existing or key not in rankings:
            rankings[key] = value

    np.save(path, rankings, allow_pickle=True)
    print(f"Saved rankings with {len(rankings)} entries to {path}")
    return rankings

## Loading MS-AA fit outputs

In [ ]:

def load_msaa_npz(path):
    data = np.load(path, allow_pickle=True)

    results_subj = data["results_subj"].tolist()
    if isinstance(results_subj, np.ndarray):
        results_subj = results_subj.tolist()

    out = {
        "K": int(data["K"]) if "K" in data else None,
        "results_subj": results_subj,
    }

    for key in ["condition_labels_str", "condition_codes", "condition_names"]:
        if key in data:
            val = data[key]
            try:
                out[key] = val.tolist()
            except Exception:
                out[key] = val

    return out


def fit_path_for_config(analysis_type, fit_scope, K, condition=None):
    if fit_scope == "across":
        return os.path.join(FIT_LOAD_DIR, f"{analysis_type}AA_across_acrossCond_K{K}.npz")
    elif fit_scope == "within":
        if condition is None:
            raise ValueError("condition is required for within-condition fits")
        return os.path.join(FIT_LOAD_DIR, f"{analysis_type}AA_within_{condition}_K{K}.npz")
    else:
        raise ValueError("FIT_SCOPE must be 'within' or 'across'")


def load_fit(analysis_type, fit_scope, K, condition=None):
    path = fit_path_for_config(analysis_type, fit_scope, K, condition=condition)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing fit file: {path}")
    fit = load_msaa_npz(path)
    fit["fit_source_file"] = path
    fit["fit_source_basename"] = os.path.basename(path)
    return fit


def get_condition_subjects_from_across_fit(fit, condition):
    labels = np.asarray(fit["condition_labels_str"])
    idx = np.where(labels == condition)[0]
    return [fit["results_subj"][i] for i in idx]


def get_subjects_for_condition(analysis_type, fit_scope, K, condition):
    if fit_scope == "across":
        fit = load_fit(analysis_type, fit_scope, K)
        return get_condition_subjects_from_across_fit(fit, condition), fit
    else:
        fit = load_fit(analysis_type, fit_scope, K, condition=condition)
        return fit["results_subj"], fit

## Reconstruction and decoding functions

In [ ]:

def to_float_array(x):
    return np.asarray(x, dtype=float)


def reconstruct_from_archetypes(sub, analysis_type, archetype_indices=None):
    sXC = to_float_array(sub["sXC"])
    S = to_float_array(sub["S"])

    if archetype_indices is None:
        idx = np.arange(S.shape[0])
    else:
        idx = np.asarray(list(archetype_indices), dtype=int)

    if analysis_type == "spatial":
        # spatial AA: sXC = T x K, S = K x V
        Xhat = sXC[:, idx] @ S[idx, :]
    elif analysis_type == "temporal":
        # temporal AA: sXC = V x K, S = K x T
        Xhat = (sXC[:, idx] @ S[idx, :]).T
    else:
        raise ValueError("analysis_type must be 'spatial' or 'temporal'")

    Xhat = (Xhat - Xhat.mean(axis=0, keepdims=True)) / (Xhat.std(axis=0, keepdims=True) + 1e-8)
    return Xhat


def build_recon_stack(subjects, analysis_type, archetype_indices=None):
    return np.stack([
        reconstruct_from_archetypes(sub, analysis_type, archetype_indices=archetype_indices)
        for sub in subjects
    ], axis=0)


def decoder(corrs):
    out = pd.DataFrame({"rank": [0.0], "accuracy": [0.0], "error": [0.0]})
    T = corrs.shape[0]

    for t in range(T):
        decoded_ind = int(np.argmax(corrs[t, :]))
        out.loc[0, "error"] += np.mean(np.abs(decoded_ind - t)) / T
        out.loc[0, "accuracy"] += (decoded_ind == t)
        out.loc[0, "rank"] += np.mean((corrs[t, :] <= corrs[t, t]).astype(int))

    out["error"] /= T
    out["accuracy"] /= T
    out["rank"] /= T
    return out


def get_xval_assignments(ndata, nfolds, rng=None):
    rng = np.random.default_rng() if rng is None else rng
    group_assignments = np.zeros(ndata, dtype=int)
    groupsize = int(np.ceil(ndata / nfolds))

    for i in range(1, nfolds):
        inds = np.arange(i * groupsize, min((i + 1) * groupsize, ndata))
        group_assignments[inds] = i

    rng.shuffle(group_assignments)
    return group_assignments


def run_timepoint_decoding(recon_stack, nfolds=2, nreps=20, seed=42):
    N, T, V = recon_stack.shape
    rng = np.random.default_rng(seed)
    rows = []

    for rep in range(nreps):
        fold_ids = get_xval_assignments(N, nfolds, rng=rng)

        for fold in range(nfolds):
            in_mask = (fold_ids == fold)
            out_mask = ~in_mask

            in_mean = recon_stack[in_mask].mean(axis=0)
            out_mean = recon_stack[out_mask].mean(axis=0)

            corrs = 1.0 - cdist(in_mean, out_mean, metric="correlation")
            corrs = np.nan_to_num(corrs, nan=0.0, posinf=0.0, neginf=0.0)

            res = decoder(corrs)
            res["rep"] = rep
            res["fold"] = fold
            rows.append(res)

    return pd.concat(rows, ignore_index=True)


def summarize_decoding_runs(dec_df):
    return {
        "mean": float(dec_df["accuracy"].mean()),
        "err": float(dec_df["accuracy"].std() / np.sqrt(max(len(dec_df), 1))),
        "std": float(dec_df["accuracy"].std()),
        "count": int(len(dec_df)),
        "rank_mean": float(dec_df["rank"].mean()),
        "error_mean": float(dec_df["error"].mean()),
    }


def decode_subjects(subjects, analysis_type, archetype_indices=None, nfolds=2, nreps=20, seed=42):
    recon_stack = build_recon_stack(subjects, analysis_type, archetype_indices=archetype_indices)
    dec_df = run_timepoint_decoding(recon_stack, nfolds=nfolds, nreps=nreps, seed=seed)
    return dec_df, summarize_decoding_runs(dec_df)

## Full reconstruction decoding

In [ ]:
if RUN_DECODING:

    def should_run_full_decoding(condition, K):
        return should_run_config(
            FULL_SUMMARY_CSV,
            FULL_KEY_COLS,
            {
                "analysis_type": ANALYSIS_TYPE,
                "fit_scope": FIT_SCOPE,
                "condition": condition,
                "K": K,
            },
            overwrite_existing=OVERWRITE_EXISTING_ROWS
        )


    full_rows_new = []

    if RUN_FULL_RECONSTRUCTION:
        for K in K_VALUES:
            for condition in CONDITIONS:
                if not should_run_full_decoding(condition, K):
                    print(f"Skipping existing full decoding: {ANALYSIS_TYPE} {FIT_SCOPE} {condition} K={K}")
                    continue

                print(f"Running full decoding: {ANALYSIS_TYPE} {FIT_SCOPE} {condition} K={K}")
                subjects, fit = get_subjects_for_condition(ANALYSIS_TYPE, FIT_SCOPE, K, condition)

                dec_df, summary = decode_subjects(
                    subjects,
                    ANALYSIS_TYPE,
                    archetype_indices=None,
                    nfolds=NFOLDS_DECODE,
                    nreps=NREPS_DECODE,
                    seed=RNG_SEED
                )

                row = {
                    "analysis_type": ANALYSIS_TYPE,
                    "fit_scope": FIT_SCOPE,
                    "condition": condition,
                    "K": K,
                    "fit_source_file": fit.get("fit_source_file", ""),
                    "fit_source_basename": fit.get("fit_source_basename", ""),
                    **summary,
                }
                full_rows_new.append(row)

    full_summary_new = pd.DataFrame(full_rows_new)
    display(full_summary_new)

    if len(full_summary_new):
        full_summary_all = append_or_update_csv(
            full_summary_new,
            FULL_SUMMARY_CSV,
            key_cols=FULL_KEY_COLS,
            overwrite_existing=OVERWRITE_EXISTING_ROWS
        )
    else:
        full_summary_all = load_existing_csv(FULL_SUMMARY_CSV)

else:
    print('RUN_DECODING=False: skipping computation in this section.')

## Per-archetype decoding

In [ ]:
if RUN_DECODING:

    def get_n_archetypes_from_fit(fit):
        sub0 = fit["results_subj"][0]
        return int(np.asarray(sub0["S"]).shape[0])


    def should_run_per_archetype_decoding(condition, K, archetype):
        return should_run_config(
            PER_ARCHETYPE_CSV,
            PER_ARCHETYPE_KEY_COLS,
            {
                "analysis_type": ANALYSIS_TYPE,
                "fit_scope": FIT_SCOPE,
                "condition": condition,
                "K": K,
                "archetype": archetype,
            },
            overwrite_existing=OVERWRITE_EXISTING_ROWS
        )


    per_arch_rows_new = []

    if RUN_PER_ARCHETYPE:
        for K in K_VALUES:
            # Load one fit to determine K_archetypes
            if FIT_SCOPE == "across":
                fit_for_shape = load_fit(ANALYSIS_TYPE, FIT_SCOPE, K)
            else:
                fit_for_shape = load_fit(ANALYSIS_TYPE, FIT_SCOPE, K, condition=CONDITIONS[0])

            n_archetypes = get_n_archetypes_from_fit(fit_for_shape)

            if ARCHETYPE_INDICES_TO_DECODE is None:
                archetype_indices = list(range(n_archetypes))
            else:
                archetype_indices = list(ARCHETYPE_INDICES_TO_DECODE)

            for condition in CONDITIONS:
                subjects, fit = get_subjects_for_condition(ANALYSIS_TYPE, FIT_SCOPE, K, condition)

                for archetype in archetype_indices:
                    if not should_run_per_archetype_decoding(condition, K, archetype):
                        continue

                    print(f"Running per-arch decoding: {ANALYSIS_TYPE} {FIT_SCOPE} {condition} K={K} arch={archetype}")

                    dec_df, summary = decode_subjects(
                        subjects,
                        ANALYSIS_TYPE,
                        archetype_indices=[archetype],
                        nfolds=NFOLDS_DECODE,
                        nreps=NREPS_DECODE,
                        seed=RNG_SEED
                    )

                    per_arch_rows_new.append({
                        "analysis_type": ANALYSIS_TYPE,
                        "fit_scope": FIT_SCOPE,
                        "condition": condition,
                        "K": K,
                        "archetype": int(archetype),
                        "fit_source_file": fit.get("fit_source_file", ""),
                        "fit_source_basename": fit.get("fit_source_basename", ""),
                        "mean_accuracy": summary["mean"],
                        "sem_accuracy": summary["err"],
                        "std_accuracy": summary["std"],
                        "count": summary["count"],
                        "rank_mean": summary["rank_mean"],
                        "error_mean": summary["error_mean"],
                    })

    per_arch_new = pd.DataFrame(per_arch_rows_new)
    display(per_arch_new.head())

    if len(per_arch_new):
        per_arch_all = append_or_update_csv(
            per_arch_new,
            PER_ARCHETYPE_CSV,
            key_cols=PER_ARCHETYPE_KEY_COLS,
            overwrite_existing=OVERWRITE_EXISTING_ROWS
        )
    else:
        per_arch_all = load_existing_csv(PER_ARCHETYPE_CSV)

else:
    print('RUN_DECODING=False: skipping computation in this section.')

## Build/update archetype rankings from per-archetype decoding

## Condition-specific ranking rule

This branch ranks archetypes separately for each condition, even for across-condition fits:

```python
rankings[(condition, K)] = condition-specific ranked archetypes
```

This replaces the older across-condition rule:

```python
rankings[K] = mean decoding across intact/word/rest
```


In [ ]:

def build_rankings_from_per_arch(per_arch_df):
    """
    Build decoding-based archetype rankings.

    IMPORTANT BRANCH CHANGE:
    For both within-condition and across-condition fits, rankings are condition-specific.

    That means:
      rankings[(condition, K)] = top archetypes for that condition at that K

    For across-condition models, the basis is shared across all conditions, but
    top-m decoding is evaluated within each condition. Therefore the top-m set
    for intact can differ from the top-m set for word and rest.
    """
    rankings = {}

    if len(per_arch_df) == 0:
        return rankings

    df = per_arch_df[
        (per_arch_df["analysis_type"].astype(str) == str(ANALYSIS_TYPE)) &
        (per_arch_df["fit_scope"].astype(str) == str(FIT_SCOPE))
    ].copy()

    # Backward-compatible column naming
    if "mean_accuracy" not in df.columns and "mean" in df.columns:
        df["mean_accuracy"] = df["mean"]

    for (condition, K), sub in df.groupby(["condition", "K"]):
        ranked = (
            sub.groupby("archetype")["mean_accuracy"]
            .mean()
            .sort_values(ascending=False)
            .index.astype(int)
            .tolist()
        )
        rankings[(str(condition), int(K))] = ranked

    return rankings


new_rankings = build_rankings_from_per_arch(per_arch_all)
rankings_all = save_or_update_rankings(
    new_rankings,
    RANKINGS_NPY,
    overwrite_existing=(OVERWRITE_EXISTING_ROWS or FORCE_REBUILD_RANKINGS)
)

print("Ranking keys:", list(rankings_all.keys())[:10])

## Top-m decoding

In [ ]:
if RUN_DECODING:

    def should_run_topm_decoding(condition, K, top_m):
        return should_run_config(
            TOPM_SUMMARY_CSV,
            TOPM_KEY_COLS,
            {
                "analysis_type": ANALYSIS_TYPE,
                "fit_scope": FIT_SCOPE,
                "condition": condition,
                "K": K,
                "top_m": top_m,
            },
            overwrite_existing=OVERWRITE_EXISTING_ROWS
        )


    def get_ranked_archetypes_for_config(rankings, condition, K):
        """
        Use condition-specific rankings for BOTH within and across fits.
        """
        key = (str(condition), int(K))
        if key not in rankings:
            raise KeyError(
                f"Missing condition-specific ranking key: {key}. "
                f"Available keys include: {list(rankings.keys())[:10]}"
            )
        return rankings[key]


    topm_rows_new = []

    if RUN_TOP_M:
        for K in K_VALUES:
            for condition in CONDITIONS:
                ranked_archetypes = get_ranked_archetypes_for_config(rankings_all, condition, K)
                subjects, fit = get_subjects_for_condition(ANALYSIS_TYPE, FIT_SCOPE, K, condition)

                for top_m in TOP_M_VALUES:
                    if not should_run_topm_decoding(condition, K, top_m):
                        print(f"Skipping existing top-m decoding: {ANALYSIS_TYPE} {FIT_SCOPE} {condition} K={K} top_m={top_m}")
                        continue

                    selected = ranked_archetypes[:top_m]

                    print(f"Running top-m decoding: {ANALYSIS_TYPE} {FIT_SCOPE} {condition} K={K} top_m={top_m} selected={selected}")

                    dec_df, summary = decode_subjects(
                        subjects,
                        ANALYSIS_TYPE,
                        archetype_indices=selected,
                        nfolds=NFOLDS_DECODE,
                        nreps=NREPS_DECODE,
                        seed=RNG_SEED
                    )

                    topm_rows_new.append({
                        "analysis_type": ANALYSIS_TYPE,
                        "fit_scope": FIT_SCOPE,
                        "condition": condition,
                        "K": K,
                        "top_m": int(top_m),
                        "selected_archetypes": str(selected),
                        "fit_source_file": fit.get("fit_source_file", ""),
                        "fit_source_basename": fit.get("fit_source_basename", ""),
                        **summary,
                    })

    topm_new = pd.DataFrame(topm_rows_new)
    display(topm_new.head())

    if len(topm_new):
        topm_all = append_or_update_csv(
            topm_new,
            TOPM_SUMMARY_CSV,
            key_cols=TOPM_KEY_COLS,
            overwrite_existing=OVERWRITE_EXISTING_ROWS
        )
    else:
        topm_all = load_existing_csv(TOPM_SUMMARY_CSV)

else:
    print('RUN_DECODING=False: skipping computation in this section.')


## Diagnostic: verify within and across fits/reconstructions differ

Run this before interpreting nearly identical within/across decoding curves. It checks whether the expected within and across fit files are being loaded, compares matrix shapes, and computes the mean absolute difference between within-fit and across-fit reconstructions for the same condition and K.

If `mean_abs_recon_diff` is exactly or nearly zero for many rows, then the within and across analyses are probably loading the same underlying fits or the saved fit files are duplicated.


In [ ]:

def quick_array_summary(arr):
    arr = np.asarray(arr, dtype=float)
    return {
        "shape": tuple(arr.shape),
        "mean": float(np.nanmean(arr)),
        "std": float(np.nanstd(arr)),
        "min": float(np.nanmin(arr)),
        "max": float(np.nanmax(arr)),
    }


def compare_within_across_for_condition(analysis_type, K, condition, max_subjects=3, archetype_indices=None):
    within_path = fit_path_for_config(analysis_type, "within", K, condition=condition)
    across_path = fit_path_for_config(analysis_type, "across", K, condition=None)

    row = {
        "analysis_type": analysis_type,
        "K": K,
        "condition": condition,
        "within_path": within_path,
        "across_path": across_path,
        "within_exists": os.path.exists(within_path),
        "across_exists": os.path.exists(across_path),
    }

    if not row["within_exists"] or not row["across_exists"]:
        return row

    within_fit = load_fit(analysis_type, "within", K, condition=condition)
    across_fit = load_fit(analysis_type, "across", K)

    within_subjects = within_fit["results_subj"]
    across_subjects = get_condition_subjects_from_across_fit(across_fit, condition)

    row["n_within_subjects"] = len(within_subjects)
    row["n_across_condition_subjects"] = len(across_subjects)

    row["within_sXC_shape"] = tuple(np.asarray(within_subjects[0]["sXC"]).shape)
    row["within_S_shape"] = tuple(np.asarray(within_subjects[0]["S"]).shape)
    row["across_sXC_shape"] = tuple(np.asarray(across_subjects[0]["sXC"]).shape)
    row["across_S_shape"] = tuple(np.asarray(across_subjects[0]["S"]).shape)

    n = min(max_subjects, len(within_subjects), len(across_subjects))
    diffs = []
    corr_vals = []

    for i in range(n):
        Xw = reconstruct_from_archetypes(within_subjects[i], analysis_type, archetype_indices=archetype_indices)
        Xa = reconstruct_from_archetypes(across_subjects[i], analysis_type, archetype_indices=archetype_indices)

        if Xw.shape != Xa.shape:
            row["shape_mismatch"] = True
            row["within_recon_shape"] = tuple(Xw.shape)
            row["across_recon_shape"] = tuple(Xa.shape)
            continue

        diffs.append(float(np.mean(np.abs(Xw - Xa))))
        corr_vals.append(float(np.corrcoef(Xw.ravel(), Xa.ravel())[0, 1]))

    row["mean_abs_recon_diff"] = float(np.mean(diffs)) if len(diffs) else np.nan
    row["recon_corr"] = float(np.mean(corr_vals)) if len(corr_vals) else np.nan

    # Also compare raw factor matrices for subject 0
    row["sXC0_mean_abs_diff"] = np.nan
    row["S0_mean_abs_diff"] = np.nan
    try:
        sxc_w = np.asarray(within_subjects[0]["sXC"], dtype=float)
        sxc_a = np.asarray(across_subjects[0]["sXC"], dtype=float)
        if sxc_w.shape == sxc_a.shape:
            row["sXC0_mean_abs_diff"] = float(np.mean(np.abs(sxc_w - sxc_a)))
        S_w = np.asarray(within_subjects[0]["S"], dtype=float)
        S_a = np.asarray(across_subjects[0]["S"], dtype=float)
        if S_w.shape == S_a.shape:
            row["S0_mean_abs_diff"] = float(np.mean(np.abs(S_w - S_a)))
    except Exception as e:
        row["factor_compare_error"] = str(e)

    return row


diagnostic_rows = []
for K in K_VALUES:
    for condition in CONDITIONS:
        diagnostic_rows.append(compare_within_across_for_condition(ANALYSIS_TYPE, K, condition, max_subjects=3))

diagnostic_df = pd.DataFrame(diagnostic_rows)
display(diagnostic_df)

# Red flags to inspect
if "mean_abs_recon_diff" in diagnostic_df.columns:
    display(diagnostic_df.sort_values("mean_abs_recon_diff").head(10))

## Quick plots

In [ ]:

COND_COLORS = {"intact": "purple", "word": "green", "rest": "black"}

def _filter_current_df(df):
    if df is None or len(df) == 0:
        return pd.DataFrame()
    out = df.copy()
    if "analysis_type" in out.columns:
        out = out[out["analysis_type"].astype(str) == str(ANALYSIS_TYPE)]
    if "fit_scope" in out.columns:
        out = out[out["fit_scope"].astype(str) == str(FIT_SCOPE)]
    if "K" in out.columns:
        out["K"] = out["K"].astype(int)
    return out.copy()


def _finish_plot(name):
    """
    Explicitly save and close. This avoids relying only on the plt.show monkey patch.
    force=True is used so these named paper/check plots are always written.
    """
    if SAVE_FIGS:
        savefig(name, force=True)
    save_current_fig()
    plt.show()
    plt.close()
def plot_full_decoding_summary(df, title=None, y_col="mean", err_col="err"):
    df = _filter_current_df(df)
    if df is None or len(df) == 0:
        print("No full decoding rows to plot.")
        return

    plt.figure(figsize=(7, 4.5))

    for condition in CONDITIONS:
        sub = df[df["condition"].astype(str) == condition].sort_values("K")
        if len(sub) == 0:
            continue

        yerr = sub[err_col] if err_col in sub.columns else None

        plt.errorbar(
            sub["K"],
            sub[y_col],
            yerr=yerr,
            marker="o",
            capsize=4,
            linewidth=2,
            color=COND_COLORS.get(condition, "gray"),
            label=condition
        )

    plt.xlabel("K")
    plt.ylabel("Decoding accuracy")
    plt.title(title or f"{ANALYSIS_TYPE} AA | {FIT_SCOPE} | full reconstruction")
    plt.legend(frameon=False)
    plt.tight_layout()
    _finish_plot(f"002_full_decoding_{ANALYSIS_TYPE}_{FIT_SCOPE}")


def plot_topm_decoding_summary(df, top_m_values=None, y_col="mean", err_col="err"):
    """
    Main requested top-m figures:
    one figure per top_m, with intact/word/rest overlaid.
    """
    df = _filter_current_df(df)
    if df is None or len(df) == 0:
        print("No top-m decoding rows to plot.")
        return

    if "top_m" not in df.columns:
        print("No top_m column found in top-m dataframe.")
        return

    df["top_m"] = df["top_m"].astype(int)

    if top_m_values is None:
        top_m_values = sorted(df["top_m"].dropna().unique())
    else:
        top_m_values = [int(x) for x in top_m_values]

    print("Top-m values available:", sorted(df["top_m"].dropna().unique()))
    print("Top-m values requested:", top_m_values)

    for top_m in top_m_values:
        sub_top = df[df["top_m"] == top_m].copy()

        if len(sub_top) == 0:
            print(f"No rows found for top_m={top_m}")
            continue

        plt.figure(figsize=(7, 4.5))

        for condition in CONDITIONS:
            sub = sub_top[sub_top["condition"].astype(str) == condition].sort_values("K")
            if len(sub) == 0:
                continue

            yerr = sub[err_col] if err_col in sub.columns else None

            plt.errorbar(
                sub["K"],
                sub[y_col],
                yerr=yerr,
                marker="o",
                capsize=4,
                linewidth=2,
                color=COND_COLORS.get(condition, "gray"),
                label=condition
            )

        plt.xlabel("K")
        plt.ylabel("Decoding accuracy")
        plt.title(f"{ANALYSIS_TYPE} AA | {FIT_SCOPE} | top-{top_m} reconstruction")
        plt.legend(frameon=False)
        plt.tight_layout()
        _finish_plot(f"002_top{top_m}_decoding_{ANALYSIS_TYPE}_{FIT_SCOPE}")


def plot_topm_all_on_one_summary(df, top_m_values=None, y_col="mean", err_col="err"):
    """
    Optional compact version:
    one figure per condition, with multiple top-m curves overlaid.
    """
    df = _filter_current_df(df)
    if df is None or len(df) == 0:
        print("No top-m decoding rows to plot.")
        return

    if "top_m" not in df.columns:
        print("No top_m column found in top-m dataframe.")
        return

    df["top_m"] = df["top_m"].astype(int)

    if top_m_values is None:
        top_m_values = sorted(df["top_m"].dropna().unique())
    else:
        top_m_values = [int(x) for x in top_m_values]

    for condition in CONDITIONS:
        plt.figure(figsize=(7, 4.5))

        for top_m in top_m_values:
            sub = df[
                (df["condition"].astype(str) == condition) &
                (df["top_m"] == top_m)
            ].sort_values("K")

            if len(sub) == 0:
                continue

            yerr = sub[err_col] if err_col in sub.columns else None

            plt.errorbar(
                sub["K"],
                sub[y_col],
                yerr=yerr,
                marker="o",
                capsize=4,
                linewidth=2,
                label=f"top-{top_m}"
            )

        plt.xlabel("K")
        plt.ylabel("Decoding accuracy")
        plt.title(f"{ANALYSIS_TYPE} AA | {FIT_SCOPE} | {condition} | top-m comparison")
        plt.legend(frameon=False)
        plt.tight_layout()
        _finish_plot(f"002_topm_comparison_{ANALYSIS_TYPE}_{FIT_SCOPE}_{condition}")


# Load existing summaries if decoding was skipped or no new rows were created.
if "full_summary_all" not in globals():
    full_summary_all = load_existing_csv(FULL_SUMMARY_CSV)

if "topm_all" not in globals():
    topm_all = load_existing_csv(TOPM_SUMMARY_CSV)

print("Full rows available:", len(full_summary_all))
print("Top-m rows available:", len(topm_all))

if len(topm_all):
    display(_filter_current_df(topm_all).head())
    if "top_m" in topm_all.columns:
        print("Available top_m values in current filtered topm_all:",
              sorted(_filter_current_df(topm_all)["top_m"].dropna().astype(int).unique()))

# Full reconstruction plot
if RUN_FIGURES:
    plot_full_decoding_summary(
        full_summary_all,
        title=f"{ANALYSIS_TYPE} AA | {FIT_SCOPE} | full reconstruction"
    )

    # One plot per top_m, intact/word/rest overlaid
    plot_topm_decoding_summary(
        topm_all,
        top_m_values=TOP_M_VALUES
    )

    # One plot per condition, multiple top-m curves overlaid
    plot_topm_all_on_one_summary(
        topm_all,
        top_m_values=TOP_M_VALUES
    )
else:
    print("RUN_FIGURES=False: skipping quick plots.")

## Audit completed outputs

In [ ]:

def audit_decoding_outputs():
    for label, path, keys in [
        ("full", FULL_SUMMARY_CSV, FULL_KEY_COLS),
        ("topm", TOPM_SUMMARY_CSV, TOPM_KEY_COLS),
        ("per_archetype", PER_ARCHETYPE_CSV, PER_ARCHETYPE_KEY_COLS),
    ]:
        df = load_existing_csv(path)
        if len(df):
            try:
                df = ensure_metadata_columns(df)
            except Exception:
                pass
        print("\n" + "="*80)
        print(label, path)
        print("="*80)

        if len(df) == 0:
            print("No rows yet.")
            continue

        print("Rows:", len(df))
        print("Columns:", list(df.columns))

        group_cols = [c for c in ["analysis_type", "fit_scope", "condition", "K"] if c in df.columns]
        if group_cols:
            display(df.groupby(group_cols).size().reset_index(name="n_rows").head(50))

        display(df.head())


audit_decoding_outputs()